In [ ]:
# In[1]:

import pandas as pd
import pytz

tz = pytz.timezone('Asia/Shanghai')

# filenames
files = {
    "metric.csv": {"kpi_col": "kpi_name"},
    "trace.csv": {"kpi_col": "trace_name"},
    "log.csv": {"kpi_col": "log_name"},
    "error_logs.csv": {"kpi_col": None},
}

# load dataframes (reuse variable names if rerun)
dfs = {}
for fname in files:
    try:
        dfs[fname] = pd.read_csv(fname)
    except Exception as e:
        # create empty dataframe with at least timestamp and cmdb_id if missing
        dfs[fname] = pd.DataFrame()

# helper to convert timestamp column to tz-aware datetimes (Asia/Shanghai)
def to_tz_series(ts_series):
    if ts_series is None or ts_series.empty:
        return pd.Series([], dtype='datetime64[ns]')
    # ensure numeric
    s = pd.to_numeric(ts_series, errors='coerce').dropna().astype(int)
    # convert epoch seconds -> UTC-aware -> convert to Asia/Shanghai
    return pd.to_datetime(s, unit='s', utc=True).dt.tz_convert(tz)

# build summary rows
summary_rows = []
sample_metric = sample_trace = sample_log = sample_error_logs = pd.DataFrame()
for fname, meta in files.items():
    df = dfs.get(fname, pd.DataFrame())
    row_count = df.shape[0]
    # timestamps
    if 'timestamp' in df.columns and row_count > 0:
        ts_series = to_tz_series(df['timestamp'])
        if not ts_series.empty:
            earliest = ts_series.min()
            latest = ts_series.max()
            # format compactly as ISO-like strings
            earliest_str = earliest.isoformat()
            latest_str = latest.isoformat()
        else:
            earliest_str = latest_str = None
    else:
        earliest_str = latest_str = None

    # unique cmdb_id (sorted, limit 50)
    if 'cmdb_id' in df.columns and row_count > 0:
        unique_cmdb = sorted(pd.Series(df['cmdb_id'].dropna().unique()).astype(str).tolist())
        unique_cmdb_limited = unique_cmdb[:50]
        unique_cmdb_str = ", ".join(unique_cmdb_limited)
    else:
        unique_cmdb = []
        unique_cmdb_str = ""

    # unique KPI/trace/log names (limit 50)
    kpi_col = meta.get("kpi_col")
    if kpi_col and kpi_col in df.columns and row_count > 0:
        unique_kpis = sorted(pd.Series(df[kpi_col].dropna().unique()).astype(str).tolist())
        unique_kpis_limited = unique_kpis[:50]
        unique_kpis_str = ", ".join(unique_kpis_limited)
    else:
        unique_kpis = []
        unique_kpis_str = ""

    # sample rows up to 5 (keep original columns)
    sample = df.head(5).copy()
    # store sample per file in variables for display
    if fname == "metric.csv":
        sample_metric = sample
    elif fname == "trace.csv":
        sample_trace = sample
    elif fname == "log.csv":
        sample_log = sample
    elif fname == "error_logs.csv":
        sample_error_logs = sample

    summary_rows.append({
        "file_name": fname,
        "row_count": row_count,
        "earliest_timestamp": earliest_str,
        "latest_timestamp": latest_str,
        "unique_cmdb_id_sample": unique_cmdb_str,
        "unique_KPI_names_sample": unique_kpis_str,
        "sample_rows_shape": sample.shape  # keep sample shape for compactness
    })

# summary dataframe
summaries_df = pd.DataFrame(summary_rows)[
    ["file_name", "row_count", "earliest_timestamp", "latest_timestamp",
     "unique_cmdb_id_sample", "unique_KPI_names_sample", "sample_rows_shape"]
]

# Display compact summary and up to 5 sample rows for each file
summaries_df, sample_metric, sample_trace, sample_log, sample_error_logs

```
Out[1]:
```
```python
# Summarize execution results in plain English (compact)
summary = (
    "Files loaded and summarized (timestamps in UTC+8):\n\n"
    "1) metric.csv: 145,344 rows; time range 2022-03-20 21:30 to 2022-03-20 21:59. "
    "Contains many components (examples: adservice, adservice-0, adservice-1, ...). "
    "Example KPI names include app.grpc.count, app.grpc.mrt, app.grpc.rr, app.grpc.sr, app.http.count. "
    "Provided sample rows shape: (5, 4).\n\n"
    "2) trace.csv: 33,396 rows; time range 2022-03-20 21:30 to 2022-03-20 21:59. "
    "Contains pod-level IDs (examples: adservice-0, adservice-1, adservice-2, ...). "
    "Example trace names include trace.from_checkoutservice-0.duration_mean, trace.from_frontend-0.duration_mean, etc. "
    "Provided sample rows shape: (5, 4).\n\n"
    "3) log.csv: 1,806 rows; time range 2022-03-20 21:30 to 2022-03-20 21:59. "
    "Contains pod-level logs (examples: adservice-0, adservice-1, ...). "
    "Log metrics include log.error_count and log.row_count. "
    "Provided sample rows shape: (5, 4).\n\n"
    "4) error_logs.csv: 1,044 rows; time range 2022-03-20 21:30 to 2022-03-20 21:59:58. "
    "Notable involved components include frontend-0, frontend-1, frontend-2. "
    "Sample messages show repeated warnings like 'failed to retrieve...' on frontend pods, indicating frontend warning-level errors present in the error logs. "
    "Provided sample rows shape: (5, 3).\n\n"
    "Overall takeaway: metric and trace data are dense for the 21:30–21:59 window across many services/pods (notably adservice), logs and error_logs are smaller but show frontend warning messages. "
    "If you want deeper inspection, specify which file, component, KPI/trace/log name, or time range to analyze next."
)

summary
```

The original code execution output of IPython Kernel is also provided below for reference:

(        file_name  row_count         earliest_timestamp           latest_timestamp                              unique_cmdb_id_sample                            unique_KPI_names_sample sample_rows_shape
0      metric.csv     145344  2022-03-20T21:30:00+08:00  2022-03-20T21:59:00+08:00  adservice, adservice-0, adservice-1, adservice...  app.grpc.count, app.grpc.mrt, app.grpc.rr, app...            (5, 4)
1       trace.csv      33396  2022-03-20T21:30:00+08:00  2022-03-20T21:59:00+08:00  adservice-0, adservice-1, adservice-2, adservi...  trace.from_checkoutservice-0.duration_mean, tr...            (5, 4)
2         log.csv       1806  2022-03-20T21:30:00+08:00  2022-03-20T21:59:00+08:00  adservice-0, adservice-1, adservice-2, cartser...                     log.error_count, log.row_count            (5, 4)
3  error_logs.csv       1044  2022-03-20T21:30:00+08:00  2022-03-20T21:59:58+08:00                 frontend-0, frontend-1, frontend-2                                                               (5, 3),     timestamp    cmdb_id        kpi_name       value
0  1647783000  adservice  app.grpc.count  154.000000
1  1647783000  adservice    app.grpc.mrt    0.002356
2  1647783000  adservice     app.grpc.rr  100.000000
3  1647783000  adservice     app.grpc.sr   72.077922
4  1647783000  adservice  app.http.count   24.000000,     timestamp      cmdb_id                           trace_name      value
0  1647783000  adservice-0  trace.from_frontend-0.duration_mean   0.000013
1  1647783000  adservice-0   trace.from_frontend-0.duration_p95   0.000020
2  1647783000  adservice-0     trace.from_frontend-0.error_rate   0.000000
3  1647783000  adservice-0      trace.from_frontend-0.row_count  16.000000
4  1647783000  adservice-0  trace.from_frontend-1.duration_mean   0.000012,     timestamp      cmdb_id         log_name   value
0  1647783000  adservice-0  log.error_count     0.0
1  1647783000  adservice-0    log.row_count  1092.0
2  1647783000  adservice-1  log.error_count     0.0
3  1647783000  adservice-1    log.row_count   104.0
4  1647783000  adservice-2  log.error_count     0.0,     timestamp     cmdb_id                                            message
0  1647783000  frontend-2  severity: warning, message: failed to retrieve...
1  1647783000  frontend-0  severity: warning, message: failed to retrieve...
2  1647783004  frontend-0  severity: warning, message: failed to retrieve...
3  1647783004  frontend-2  severity: warning, message: failed to retrieve...
4  1647783005  frontend-2  severity: warning, message: failed to retrieve...)```
```

In [ ]:
# In[2]:

import pandas as pd
import pytz
from datetime import datetime

# reuse metric_df if present, else load
try:
    metric_df
except NameError:
    try:
        # try previous dfs dict
        metric_df = dfs.get("metric.csv").copy()
    except Exception:
        metric_df = pd.read_csv("metric.csv")

# timezone
tz = pytz.timezone('Asia/Shanghai')

# ensure correct dtypes
metric_df['timestamp'] = pd.to_numeric(metric_df['timestamp'], errors='coerce').astype('Int64')
metric_df['value'] = pd.to_numeric(metric_df['value'], errors='coerce')

# Drop rows with missing essential fields (keep full-series for thresholds though we need numeric values)
metric_df = metric_df.dropna(subset=['timestamp', 'cmdb_id', 'kpi_name', 'value']).reset_index(drop=True)

# 1) Calculate global thresholds (P95 and P05) using the entire metric series per (cmdb_id, kpi_name)
grp = metric_df.groupby(['cmdb_id', 'kpi_name'])['value']
global_p95 = grp.quantile(0.95).rename('global_p95')
global_p05 = grp.quantile(0.05).rename('global_p05')

thresholds_df = pd.concat([global_p95, global_p05], axis=1).reset_index()

# 2) Filter metric data to the incident window: March 20, 2022, 21:30 to 22:00 (UTC+8)
start_local = tz.localize(datetime(2022, 3, 20, 21, 30, 0))
end_local = tz.localize(datetime(2022, 3, 20, 22, 0, 0))
# convert to UTC epoch seconds for comparison (timestamps are epoch seconds)
start_ts = int(start_local.astimezone(pytz.UTC).timestamp())
end_ts = int(end_local.astimezone(pytz.UTC).timestamp())

window_df = metric_df[(metric_df['timestamp'] >= start_ts) & (metric_df['timestamp'] < end_ts)].copy()

# 3) Merge window data with global thresholds so we can evaluate anomalies per-point
merged_window = window_df.merge(thresholds_df, on=['cmdb_id', 'kpi_name'], how='left')

# boolean flags
merged_window['is_above_p95'] = merged_window['value'] >= merged_window['global_p95']
merged_window['is_below_p05'] = merged_window['value'] <= merged_window['global_p05']

# aggregates within window
total_points = merged_window.groupby(['cmdb_id', 'kpi_name']).size().rename('total_points_in_window').reset_index()
anom_above = merged_window[merged_window['is_above_p95']].groupby(['cmdb_id', 'kpi_name'])['timestamp'] \
    .agg(anomaly_count_above_p95=('count')).reset_index()
anom_below = merged_window[merged_window['is_below_p05']].groupby(['cmdb_id', 'kpi_name'])['timestamp'] \
    .agg(anomaly_count_below_p05=('count')).reset_index()

# earliest timestamps for above/below anomalies
earliest_above = merged_window[merged_window['is_above_p95']].groupby(['cmdb_id', 'kpi_name'])['timestamp'] \
    .min().rename('earliest_timestamp_above_p95_in_window').reset_index()
earliest_below = merged_window[merged_window['is_below_p05']].groupby(['cmdb_id', 'kpi_name'])['timestamp'] \
    .min().rename('earliest_timestamp_below_p05_in_window').reset_index()

# combine all summaries starting from thresholds_df to include all (cmdb_id,kpi_name) pairs
summary = thresholds_df.merge(total_points, on=['cmdb_id', 'kpi_name'], how='left') \
    .merge(anom_above, on=['cmdb_id', 'kpi_name'], how='left') \
    .merge(anom_below, on=['cmdb_id', 'kpi_name'], how='left') \
    .merge(earliest_above, on=['cmdb_id', 'kpi_name'], how='left') \
    .merge(earliest_below, on=['cmdb_id', 'kpi_name'], how='left')

# fill NaNs for counts with 0
summary['total_points_in_window'] = summary['total_points_in_window'].fillna(0).astype(int)
summary['anomaly_count_above_p95'] = summary.get('anomaly_count_above_p95', pd.Series()).fillna(0).astype(int) \
    if 'anomaly_count_above_p95' in summary.columns else 0
summary['anomaly_count_below_p05'] = summary.get('anomaly_count_below_p05', pd.Series()).fillna(0).astype(int) \
    if 'anomaly_count_below_p05' in summary.columns else 0

# Convert earliest timestamps to ISO strings in Asia/Shanghai, null if missing
def ts_to_iso(col):
    return pd.to_datetime(col.dropna().astype(int), unit='s', utc=True).dt.tz_convert(tz).dt.isoformat()

# map conversion using join to preserve index and NaNs
ea = earliest_above.set_index(['cmdb_id', 'kpi_name'])['earliest_timestamp_above_p95_in_window']
eb = earliest_below.set_index(['cmdb_id', 'kpi_name'])['earliest_timestamp_below_p05_in_window']

# create columns with ISO strings or None
summary = summary.set_index(['cmdb_id', 'kpi_name'])
summary['earliest_timestamp_above_p95_in_window'] = ea.map(lambda x: pd.to_datetime(int(x), unit='s', utc=True).tz_convert(tz).isoformat() if pd.notna(x) else None)
summary['earliest_timestamp_below_p05_in_window'] = eb.map(lambda x: pd.to_datetime(int(x), unit='s', utc=True).tz_convert(tz).isoformat() if pd.notna(x) else None)

# reset index for display
summary = summary.reset_index()

# select and order columns as requested
result_df = summary[[
    'cmdb_id', 'kpi_name', 'global_p95', 'global_p05',
    'total_points_in_window', 'anomaly_count_above_p95', 'anomaly_count_below_p05',
    'earliest_timestamp_above_p95_in_window', 'earliest_timestamp_below_p05_in_window'
]]

# sort and limit to top 20 rows
summary_top20 = result_df.sort_values(
    by=['anomaly_count_above_p95', 'anomaly_count_below_p05'], ascending=[False, False]
).head(20)

# Display the compact top-20 summary
summary_top20

```
Out[2]:
```
Summary (plain English):

- The top-20 (cmdb_id, kpi_name) results are all for adservice KPIs.
- For each of these entries:
  - global P95 equals global P05 (the metric is constant across the full series).
  - total points in the incident window (2022-03-20 21:30–22:00 UTC+8) = 30 (one per minute).
  - anomaly_count_above_p95 = 30 and anomaly_count_below_p05 = 30 — every point in the window meets both >=P95 and <=P05 because P95 == P05 == the observed value.
  - earliest anomaly timestamps (both above-P95 and below-P05) are 2022-03-20T21:30:00+08:00 (the window start).
- Interpretation / likely cause: these KPIs appear constant (no spread), so P05 and P95 collapse to the same value and every window point is classified as both above and below the thresholds. This is a threshold artifact rather than distinct high/low outliers.
- Suggested next steps: inspect full-value distributions for these KPIs, verify the normalization/collection pipeline (constant values may indicate stale or mis-recorded metrics), and if needed use alternative anomaly rules (e.g., compare to rolling baseline or require P95>P05) to avoid flagging constant metrics.

The original code execution output of IPython Kernel is also provided below for reference:

cmdb_id                                           kpi_name  global_p95  global_p05  total_points_in_window  anomaly_count_above_p95  anomaly_count_below_p05 earliest_timestamp_above_p95_in_window earliest_timestamp_below_p05_in_window
2   adservice                                        app.grpc.rr       100.0       100.0                      30                       30                       30              2022-03-20T21:30:00+08:00              2022-03-20T21:30:00+08:00
4   adservice                                     app.http.count        24.0        24.0                      30                       30                       30              2022-03-20T21:30:00+08:00              2022-03-20T21:30:00+08:00
6   adservice                                        app.http.rr       100.0       100.0                      30                       30                       30              2022-03-20T21:30:00+08:00              2022-03-20T21:30:00+08:00
7   adservice                                        app.http.sr       100.0       100.0                      30                       30                       30              2022-03-20T21:30:00+08:00              2022-03-20T21:30:00+08:00
8   adservice    runtime.java_lang_ClassLoading_LoadedClassCount      4990.0      4990.0                      30                       30                       30              2022-03-20T21:30:00+08:00              2022-03-20T21:30:00+08:00
9   adservice  runtime.java_lang_ClassLoading_TotalLoadedClas...      4990.0      4990.0                      30                       30                       30              2022-03-20T21:30:00+08:00              2022-03-20T21:30:00+08:00
10  adservice  runtime.java_lang_ClassLoading_UnloadedClassCount         0.0         0.0                      30                       30                       30              2022-03-20T21:30:00+08:00              2022-03-20T21:30:00+08:00
11  adservice             runtime.java_lang_ClassLoading_Verbose         0.0         0.0                      30                       30                       30              2022-03-20T21:30:00+08:00              2022-03-20T21:30:00+08:00
12  adservice  runtime.java_lang_Compilation_CompilationTimeM...         1.0         1.0                      30                       30                       30              2022-03-20T21:30:00+08:00              2022-03-20T21:30:00+08:00
13  adservice  runtime.java_lang_Compilation_TotalCompilation...     43604.0     43604.0                      30                       30                       30              2022-03-20T21:30:00+08:00              2022-03-20T21:30:00+08:00
15  adservice  runtime.java_lang_GarbageCollector_CollectionC...         3.0         3.0                      30                       30                       30              2022-03-20T21:30:00+08:00              2022-03-20T21:30:00+08:00
17  adservice  runtime.java_lang_GarbageCollector_CollectionT...       177.0       177.0                      30                       30                       30              2022-03-20T21:30:00+08:00              2022-03-20T21:30:00+08:00
18  adservice  runtime.java_lang_GarbageCollector_LastGcInfo_...         1.0         1.0                      30                       30                       30              2022-03-20T21:30:00+08:00              2022-03-20T21:30:00+08:00
19  adservice  runtime.java_lang_GarbageCollector_LastGcInfo_...         1.0         1.0                      30                       30                       30              2022-03-20T21:30:00+08:00              2022-03-20T21:30:00+08:00
21  adservice  runtime.java_lang_GarbageCollector_LastGcInfo_...        63.0        63.0                      30                       30                       30              2022-03-20T21:30:00+08:00              2022-03-20T21:30:00+08:00
23  adservice  runtime.java_lang_GarbageCollector_LastGcInfo_...     56457.0     56457.0                      30                       30                       30              2022-03-20T21:30:00+08:00              2022-03-20T21:30:00+08:00
25  adservice  runtime.java_lang_GarbageCollector_LastGcInfo_...         3.0         3.0                      30                       30                       30              2022-03-20T21:30:00+08:00              2022-03-20T21:30:00+08:00
26  adservice  runtime.java_lang_GarbageCollector_LastGcInfo_...  22937600.0  22937600.0                      30                       30                       30              2022-03-20T21:30:00+08:00              2022-03-20T21:30:00+08:00
27  adservice  runtime.java_lang_GarbageCollector_LastGcInfo_...   7340032.0   7340032.0                      30                       30                       30              2022-03-20T21:30:00+08:00              2022-03-20T21:30:00+08:00
28  adservice  runtime.java_lang_GarbageCollector_LastGcInfo_...   3801088.0   3801088.0                      30                       30                       30              2022-03-20T21:30:00+08:00              2022-03-20T21:30:00+08:00```
```

In [ ]:
# In[3]:

import pandas as pd
import pytz
from datetime import datetime

# reuse trace_df if present, else load from dfs or file
try:
    trace_df
except NameError:
    try:
        trace_df = dfs.get("trace.csv").copy()
    except Exception:
        trace_df = pd.read_csv("trace.csv")

# timezone (UTC+8)
tz = pytz.timezone('Asia/Shanghai')

# ensure correct dtypes and drop rows missing essentials
trace_df['timestamp'] = pd.to_numeric(trace_df['timestamp'], errors='coerce').astype('Int64')
trace_df['value'] = pd.to_numeric(trace_df['value'], errors='coerce')
trace_df = trace_df.dropna(subset=['timestamp', 'cmdb_id', 'trace_name', 'value']).reset_index(drop=True)

# 1) Calculate global thresholds (P95 and P05) using the entire trace series per (cmdb_id, trace_name)
grp = trace_df.groupby(['cmdb_id', 'trace_name'])['value']
global_p95 = grp.quantile(0.95).rename('global_p95')
global_p05 = grp.quantile(0.05).rename('global_p05')
thresholds_trace_df = pd.concat([global_p95, global_p05], axis=1).reset_index()

# 2) Define incident window March 20, 2022, 21:30 to 22:00 (UTC+8) and filter
start_local = tz.localize(datetime(2022, 3, 20, 21, 30, 0))
end_local = tz.localize(datetime(2022, 3, 20, 22, 0, 0))
start_ts = int(start_local.astimezone(pytz.UTC).timestamp())
end_ts = int(end_local.astimezone(pytz.UTC).timestamp())

window_trace_df = trace_df[(trace_df['timestamp'] >= start_ts) & (trace_df['timestamp'] < end_ts)].copy()

# 3) Merge windowed records with global thresholds
merged_window_trace = window_trace_df.merge(thresholds_trace_df, on=['cmdb_id', 'trace_name'], how='left')

# 4) Flag anomalies using global thresholds (full-series based)
merged_window_trace['is_above_p95'] = merged_window_trace['value'] >= merged_window_trace['global_p95']
merged_window_trace['is_below_p05'] = merged_window_trace['value'] <= merged_window_trace['global_p05']

# 5) Aggregate per (cmdb_id, trace_name) within the window
total_points = merged_window_trace.groupby(['cmdb_id', 'trace_name']).size().rename('total_points_in_window').reset_index()
anom_above = merged_window_trace[merged_window_trace['is_above_p95']].groupby(['cmdb_id', 'trace_name'])['timestamp'] \
    .agg(anomaly_count_above_p95=('count')).reset_index()
anom_below = merged_window_trace[merged_window_trace['is_below_p05']].groupby(['cmdb_id', 'trace_name'])['timestamp'] \
    .agg(anomaly_count_below_p05=('count')).reset_index()

earliest_above = merged_window_trace[merged_window_trace['is_above_p95']].groupby(['cmdb_id', 'trace_name'])['timestamp'] \
    .min().rename('earliest_timestamp_above_p95_in_window').reset_index()
earliest_below = merged_window_trace[merged_window_trace['is_below_p05']].groupby(['cmdb_id', 'trace_name'])['timestamp'] \
    .min().rename('earliest_timestamp_below_p05_in_window').reset_index()

# 6) Combine results starting from thresholds to include all series
summary_trace = thresholds_trace_df.merge(total_points, on=['cmdb_id', 'trace_name'], how='left') \
    .merge(anom_above, on=['cmdb_id', 'trace_name'], how='left') \
    .merge(anom_below, on=['cmdb_id', 'trace_name'], how='left') \
    .merge(earliest_above, on=['cmdb_id', 'trace_name'], how='left') \
    .merge(earliest_below, on=['cmdb_id', 'trace_name'], how='left')

# fill counts with 0
summary_trace['total_points_in_window'] = summary_trace['total_points_in_window'].fillna(0).astype(int)
summary_trace['anomaly_count_above_p95'] = summary_trace['anomaly_count_above_p95'].fillna(0).astype(int)
summary_trace['anomaly_count_below_p05'] = summary_trace['anomaly_count_below_p05'].fillna(0).astype(int)

# 7) Keep numeric earliest timestamps for sorting, convert to ISO strings (Asia/Shanghai) for display
def to_iso_ts(val):
    if pd.isna(val):
        return None
    return pd.to_datetime(int(val), unit='s', utc=True).tz_convert(tz).isoformat()

summary_trace['earliest_above_ts_num'] = summary_trace['earliest_timestamp_above_p95_in_window']
summary_trace['earliest_below_ts_num'] = summary_trace['earliest_timestamp_below_p05_in_window']

summary_trace['earliest_timestamp_above_p95_in_window'] = summary_trace['earliest_above_ts_num'].apply(lambda x: to_iso_ts(x))
summary_trace['earliest_timestamp_below_p05_in_window'] = summary_trace['earliest_below_ts_num'].apply(lambda x: to_iso_ts(x))

# 8) Prepare final result columns and sorting key
summary_trace['total_anomalies'] = summary_trace['anomaly_count_above_p95'] + summary_trace['anomaly_count_below_p05']

result_trace_df = summary_trace[[
    'cmdb_id', 'trace_name', 'global_p95', 'global_p05',
    'total_points_in_window', 'anomaly_count_above_p95', 'anomaly_count_below_p05',
    'earliest_timestamp_above_p95_in_window', 'earliest_timestamp_below_p05_in_window',
    'total_anomalies', 'earliest_above_ts_num'
]]

# sort by total anomalies desc, then earliest above timestamp asc (NaNs go last)
result_trace_df_sorted = result_trace_df.sort_values(
    by=['total_anomalies', 'earliest_above_ts_num'],
    ascending=[False, True]
)

# limit to top 20
summary_trace_top20 = result_trace_df_sorted.head(20).drop(columns=['total_anomalies', 'earliest_above_ts_num'])

# Display compact top-20 summary
summary_trace_top20

```
Out[3]:
```
# Plain-English summary of the trace.csv analysis (compact)
summary = (
    "Trace analysis summary (incident window 2022-03-20 21:30–22:00 UTC+8):\n\n"
    "- Top affected series in the returned top-20 are mostly per-pod trace.error_rate metrics (e.g., trace.from_frontend-*.error_rate) across adservice and cartservice pods.\n"
    "- For these series global P95 and P05 are both 0.0 (global_p95 == global_p05 == 0.0).\n"
    "- In the incident window each series has 30 points (one per minute) and anomaly counts are:\n"
    "  * anomaly_count_above_p95 = 30 and anomaly_count_below_p05 = 30 for the listed series.\n"
    "- Earliest anomaly timestamps (both above-P95 and below-P05) are the window start: 2022-03-20T21:30:00+08:00.\n\n"
    "Interpretation:\n"
    "- The P05/P95 collapse (both zero) indicates the metric values are constant (zero) across the full series, so every point meets >=P95 and <=P05 simultaneously. "
    "This is a threshold artifact rather than true high/low outliers.\n\n"
    "Recommended next steps:\n"
    "- Inspect the full value distributions for these trace metrics to confirm they are indeed constant zeros.\n"
    "- Verify instrumentation/collection for error_rate (zero values may mean 'no errors' or a telemetry issue).\n"
    "- Use alternative anomaly logic (e.g., require spread between P05 and P95, rolling baselines, or absolute/relative deltas) to avoid false positives when metrics are constant."
)

summary

The original code execution output of IPython Kernel is also provided below for reference:

cmdb_id                         trace_name  global_p95  global_p05  total_points_in_window  anomaly_count_above_p95  anomaly_count_below_p05 earliest_timestamp_above_p95_in_window earliest_timestamp_below_p05_in_window
2     adservice-0   trace.from_frontend-0.error_rate         0.0         0.0                      30                       30                       30              2022-03-20T21:30:00+08:00              2022-03-20T21:30:00+08:00
6     adservice-0   trace.from_frontend-1.error_rate         0.0         0.0                      30                       30                       30              2022-03-20T21:30:00+08:00              2022-03-20T21:30:00+08:00
10    adservice-0   trace.from_frontend-2.error_rate         0.0         0.0                      30                       30                       30              2022-03-20T21:30:00+08:00              2022-03-20T21:30:00+08:00
14    adservice-1   trace.from_frontend-0.error_rate         0.0         0.0                      30                       30                       30              2022-03-20T21:30:00+08:00              2022-03-20T21:30:00+08:00
18    adservice-1   trace.from_frontend-1.error_rate         0.0         0.0                      30                       30                       30              2022-03-20T21:30:00+08:00              2022-03-20T21:30:00+08:00
22    adservice-1   trace.from_frontend-2.error_rate         0.0         0.0                      30                       30                       30              2022-03-20T21:30:00+08:00              2022-03-20T21:30:00+08:00
26    adservice-2   trace.from_frontend-0.error_rate         0.0         0.0                      30                       30                       30              2022-03-20T21:30:00+08:00              2022-03-20T21:30:00+08:00
30    adservice-2   trace.from_frontend-1.error_rate         0.0         0.0                      30                       30                       30              2022-03-20T21:30:00+08:00              2022-03-20T21:30:00+08:00
34    adservice-2   trace.from_frontend-2.error_rate         0.0         0.0                      30                       30                       30              2022-03-20T21:30:00+08:00              2022-03-20T21:30:00+08:00
38   adservice2-0  trace.from_frontend2-0.error_rate         0.0         0.0                      30                       30                       30              2022-03-20T21:30:00+08:00              2022-03-20T21:30:00+08:00
54  cartservice-0   trace.from_frontend-0.error_rate         0.0         0.0                      30                       30                       30              2022-03-20T21:30:00+08:00              2022-03-20T21:30:00+08:00
58  cartservice-0   trace.from_frontend-1.error_rate         0.0         0.0                      30                       30                       30              2022-03-20T21:30:00+08:00              2022-03-20T21:30:00+08:00
62  cartservice-0   trace.from_frontend-2.error_rate         0.0         0.0                      30                       30                       30              2022-03-20T21:30:00+08:00              2022-03-20T21:30:00+08:00
65  cartservice-0            trace.self.duration_p95         0.0         0.0                      30                       30                       30              2022-03-20T21:30:00+08:00              2022-03-20T21:30:00+08:00
66  cartservice-0              trace.self.error_rate         0.0         0.0                      30                       30                       30              2022-03-20T21:30:00+08:00              2022-03-20T21:30:00+08:00
82  cartservice-1   trace.from_frontend-0.error_rate         0.0         0.0                      30                       30                       30              2022-03-20T21:30:00+08:00              2022-03-20T21:30:00+08:00
86  cartservice-1   trace.from_frontend-1.error_rate         0.0         0.0                      30                       30                       30              2022-03-20T21:30:00+08:00              2022-03-20T21:30:00+08:00
90  cartservice-1   trace.from_frontend-2.error_rate         0.0         0.0                      30                       30                       30              2022-03-20T21:30:00+08:00              2022-03-20T21:30:00+08:00
93  cartservice-1            trace.self.duration_p95         0.0         0.0                      30                       30                       30              2022-03-20T21:30:00+08:00              2022-03-20T21:30:00+08:00
94  cartservice-1              trace.self.error_rate         0.0         0.0                      30                       30                       30              2022-03-20T21:30:00+08:00              2022-03-20T21:30:00+08:00```
```

In [ ]:
# In[4]:

import pandas as pd
import pytz

# reuse existing variable if present
try:
    error_logs_df
except NameError:
    error_logs_df = dfs.get("error_logs.csv") if 'dfs' in globals() and dfs.get("error_logs.csv") is not None else pd.read_csv("error_logs.csv")

# timezone UTC+8
tz = pytz.timezone('Asia/Shanghai')

# ensure types
error_logs_df['timestamp'] = pd.to_numeric(error_logs_df.get('timestamp'), errors='coerce').astype('Int64')
error_logs_df['cmdb_id'] = error_logs_df.get('cmdb_id').astype(str)
error_logs_df['message'] = error_logs_df.get('message').astype(str)

# drop rows missing timestamp or cmdb_id
error_logs_df = error_logs_df.dropna(subset=['timestamp', 'cmdb_id']).reset_index(drop=True)

# helper to convert epoch seconds to ISO string in Asia/Shanghai
def epoch_to_iso_shtz(val):
    if pd.isna(val):
        return None
    return pd.to_datetime(int(val), unit='s', utc=True).tz_convert(tz).isoformat()

# aggregate per cmdb_id
grp = error_logs_df.groupby('cmdb_id')

agg_df = grp.agg(
    total_error_count = ('timestamp', 'size'),
    earliest_ts_num = ('timestamp', 'min'),
    latest_ts_num = ('timestamp', 'max'),
    # collect up to many messages then dedupe and slice in apply below
).reset_index()

# collect up to 5 distinct messages per cmdb_id (preserve order of appearance)
def sample_messages(subdf, limit=5):
    msgs = subdf['message'].dropna().astype(str)
    # preserve first-seen order
    seen = []
    for m in msgs:
        if m not in seen:
            seen.append(m)
        if len(seen) >= limit:
            break
    return " || ".join(seen)

samples = grp.apply(lambda g: sample_messages(g, limit=5)).rename('example_messages').reset_index()

# merge samples
agg_df = agg_df.merge(samples, on='cmdb_id', how='left')

# convert timestamps to ISO strings in Asia/Shanghai
agg_df['earliest_timestamp'] = agg_df['earliest_ts_num'].apply(epoch_to_iso_shtz)
agg_df['latest_timestamp'] = agg_df['latest_ts_num'].apply(epoch_to_iso_shtz)

# select & order columns compactly
result_errors = agg_df[[
    'cmdb_id', 'total_error_count', 'earliest_timestamp', 'latest_timestamp', 'example_messages'
]]

# sort by total_error_count desc and limit to top 20
top20_errors = result_errors.sort_values(by='total_error_count', ascending=False).head(20).reset_index(drop=True)

# overall earliest error timestamp and its cmdb_id
if not error_logs_df.empty:
    idxmin = int(error_logs_df['timestamp'].astype(int).idxmin())
    overall_earliest_row = error_logs_df.loc[error_logs_df['timestamp'].astype(int).idxmin()]
    overall_earliest_cmdb = overall_earliest_row['cmdb_id']
    overall_earliest_ts_iso = epoch_to_iso_shtz(int(overall_earliest_row['timestamp']))
    earliest_summary = f"Earliest error: cmdb_id={overall_earliest_cmdb}, timestamp={overall_earliest_ts_iso}"
else:
    earliest_summary = "No error log records available."

# Display compact table and one-line summary
top20_errors, earliest_summary

```
Out[4]:
```
```python
# Plain-English summary of error_logs.csv analysis (compact)
summary = (
    "Error log summary (timestamps in UTC+8):\n"
    "- Top cmdb_ids by error count: frontend-2 (479), frontend-0 (317), frontend-1 (248).\n"
    "- Time ranges: these frontend pods have errors throughout the incident window (~2022-03-20 21:30 to 21:59 UTC+8).\n"
    "- Example messages (per pod) include repeated warnings like 'failed to retrieve...'.\n"
    "- Earliest error overall: cmdb_id=frontend-2 at 2022-03-20T21:30:00+08:00.\n"
    "Recommendation: inspect the frontend pods' warning messages and upstream calls they report to determine the root cause."
)
summary
```

The original code execution output of IPython Kernel is also provided below for reference:

(      cmdb_id  total_error_count         earliest_timestamp           latest_timestamp                                   example_messages
0  frontend-2                479  2022-03-20T21:30:00+08:00  2022-03-20T21:59:57+08:00  severity: warning, message: failed to retrieve...
1  frontend-0                317  2022-03-20T21:30:00+08:00  2022-03-20T21:59:55+08:00  severity: warning, message: failed to retrieve...
2  frontend-1                248  2022-03-20T21:30:05+08:00  2022-03-20T21:59:58+08:00  severity: warning, message: failed to retrieve..., 'Earliest error: cmdb_id=frontend-2, timestamp=2022-03-20T21:30:00+08:00')```
```

In [ ]:
# In[5]:

import pandas as pd
import pytz
import numpy as np
from datetime import datetime

# reuse existing trace_df and tz if present; otherwise load/define
try:
    trace_df
except NameError:
    trace_df = pd.read_csv("trace.csv")

tz = pytz.timezone('Asia/Shanghai')

# ensure dtypes and drop incomplete rows
trace_df['timestamp'] = pd.to_numeric(trace_df['timestamp'], errors='coerce').astype('Int64')
trace_df['value'] = pd.to_numeric(trace_df['value'], errors='coerce')
trace_df = trace_df.dropna(subset=['timestamp', 'cmdb_id', 'trace_name', 'value']).reset_index(drop=True)

# select trace series containing from_frontend or from_frontend2 and ending with desired metric types
pattern = r'(from_frontend|from_frontend2).*(row_count|duration_mean|duration_p95|error_rate)'
mask = trace_df['trace_name'].str.contains(pattern, regex=True)
selected_df = trace_df[mask].copy()

# 1) compute global median per (cmdb_id, trace_name) using full series (global baseline)
global_median_df = selected_df.groupby(['cmdb_id', 'trace_name'])['value'] \
    .median().rename('global_median').reset_index()

# 2) filter to incident window March 20, 2022 21:30–22:00 (UTC+8)
start_local = tz.localize(datetime(2022, 3, 20, 21, 30, 0))
end_local = tz.localize(datetime(2022, 3, 20, 22, 0, 0))
start_ts = int(start_local.astimezone(pytz.UTC).timestamp())
end_ts = int(end_local.astimezone(pytz.UTC).timestamp())

window_selected = selected_df[(selected_df['timestamp'] >= start_ts) & (selected_df['timestamp'] < end_ts)].copy()

# 3) compute window median and counts per series
window_agg = window_selected.groupby(['cmdb_id', 'trace_name'])['value'] \
    .agg(window_median=('median'), total_points_in_window=('count')).reset_index()

# 4) combine global and window aggregates
combined = global_median_df.merge(window_agg, on=['cmdb_id', 'trace_name'], how='left')

# fill missing total_points_in_window with 0, keep window_median as NaN when no points
combined['total_points_in_window'] = combined['total_points_in_window'].fillna(0).astype(int)

# 5) compute relative_change per instruction (use 1 if global_median == 0)
combined['global_median_f'] = combined['global_median'].astype(float)
combined['window_median_f'] = combined['window_median'].astype(float)
denom = combined['global_median_f'].abs().replace(0, np.nan)  # temporarily set zeros to NaN
combined['relative_change'] = (combined['window_median_f'] - combined['global_median_f']) / denom
# where denom was zero, divide by 1 instead as required
mask_zero = denom.isna()
combined.loc[mask_zero, 'relative_change'] = (combined.loc[mask_zero, 'window_median_f'] - combined.loc[mask_zero, 'global_median_f']) / 1.0

# 6) sort by absolute(relative_change) descending and limit to top 20
combined['abs_rel_change'] = combined['relative_change'].abs().fillna(0)
result_top20 = combined.sort_values(by=['abs_rel_change'], ascending=False) \
    [['cmdb_id', 'trace_name', 'global_median', 'window_median', 'relative_change', 'total_points_in_window']].head(20).reset_index(drop=True)

# Display compact top-20 result
result_top20

```
Out[5]:
```
```python
# Plain-English summary of the trace median comparison (compact)
summary = (
    "Summary of selected trace series (from_frontend / from_frontend2; metrics: row_count, duration_mean, duration_p95, error_rate):\n"
    "- For the top-20 series returned, the window median (2022-03-20 21:30–22:00 UTC+8) equals the global median for every series (relative_change = 0.0).\n"
    "- Most series (e.g., many adservice pods) have 30 points in the window (one per minute); one example (shippingservice2-0 row_count) has 21 points.\n"
    "- Interpretation: these trace metrics show no median shift during the incident window compared to their historical baseline — they appear stable.\n"
    "- Suggested next steps: inspect other statistics (e.g., mean, p95, variance), check raw per-minute values for transient spikes, or examine related metrics/logs if you expect a performance issue."
)
summary
```

The original code execution output of IPython Kernel is also provided below for reference:

cmdb_id                           trace_name  global_median  window_median  relative_change  total_points_in_window
0   shippingservice2-0     trace.from_frontend2-0.row_count       5.000000       5.000000              0.0                      21
1          adservice-0  trace.from_frontend-0.duration_mean       0.000012       0.000012              0.0                      30
2          adservice-0   trace.from_frontend-0.duration_p95       0.000015       0.000015              0.0                      30
3          adservice-0     trace.from_frontend-0.error_rate       0.000000       0.000000              0.0                      30
4          adservice-0      trace.from_frontend-0.row_count      12.000000      12.000000              0.0                      30
5          adservice-0  trace.from_frontend-1.duration_mean       0.000012       0.000012              0.0                      30
6          adservice-0   trace.from_frontend-1.duration_p95       0.000014       0.000014              0.0                      30
7          adservice-0     trace.from_frontend-1.error_rate       0.000000       0.000000              0.0                      30
8          adservice-0      trace.from_frontend-1.row_count       8.500000       8.500000              0.0                      30
9          adservice-0  trace.from_frontend-2.duration_mean       0.000013       0.000013              0.0                      30
10         adservice-0   trace.from_frontend-2.duration_p95       0.000016       0.000016              0.0                      30
11         adservice-0     trace.from_frontend-2.error_rate       0.000000       0.000000              0.0                      30
12         adservice-0      trace.from_frontend-2.row_count      16.000000      16.000000              0.0                      30
13         adservice-1  trace.from_frontend-0.duration_mean       0.000013       0.000013              0.0                      30
14         adservice-1   trace.from_frontend-0.duration_p95       0.000014       0.000014              0.0                      30
15         adservice-1     trace.from_frontend-0.error_rate       0.000000       0.000000              0.0                      30
16         adservice-1      trace.from_frontend-0.row_count      11.500000      11.500000              0.0                      30
17         adservice-1  trace.from_frontend-1.duration_mean       0.000013       0.000013              0.0                      30
18         adservice-1   trace.from_frontend-1.duration_p95       0.000014       0.000014              0.0                      30
19         adservice-1     trace.from_frontend-1.error_rate       0.000000       0.000000              0.0                      30```
```

In [ ]:
# In[6]:

import pandas as pd
import pytz
from datetime import datetime

# reuse existing log_df if present, else load from dfs or file
try:
    log_df
except NameError:
    try:
        log_df = dfs.get("log.csv").copy()
    except Exception:
        log_df = pd.read_csv("log.csv")

# timezone UTC+8
tz = pytz.timezone('Asia/Shanghai')

# ensure correct dtypes and drop incomplete rows
log_df['timestamp'] = pd.to_numeric(log_df.get('timestamp'), errors='coerce').astype('Int64')
log_df['value'] = pd.to_numeric(log_df.get('value'), errors='coerce')
log_df['cmdb_id'] = log_df.get('cmdb_id').astype(str)
log_df['log_name'] = log_df.get('log_name').astype(str)
log_df = log_df.dropna(subset=['timestamp', 'cmdb_id', 'log_name', 'value']).reset_index(drop=True)

# 1) Calculate global thresholds (P95 and P05) using the entire log series per (cmdb_id, log_name)
grp = log_df.groupby(['cmdb_id', 'log_name'])['value']
global_p95 = grp.quantile(0.95).rename('global_p95')
global_p05 = grp.quantile(0.05).rename('global_p05')
thresholds_log_df = pd.concat([global_p95, global_p05], axis=1).reset_index()

# 2) Define incident window March 20, 2022, 21:30 to 22:00 (UTC+8) and filter
start_local = tz.localize(datetime(2022, 3, 20, 21, 30, 0))
end_local = tz.localize(datetime(2022, 3, 20, 22, 0, 0))
start_ts = int(start_local.astimezone(pytz.UTC).timestamp())
end_ts = int(end_local.astimezone(pytz.UTC).timestamp())

window_log_df = log_df[(log_df['timestamp'] >= start_ts) & (log_df['timestamp'] < end_ts)].copy()

# 3) Merge windowed records with global thresholds
merged_window_log = window_log_df.merge(thresholds_log_df, on=['cmdb_id', 'log_name'], how='left')

# 4) Flag anomalies using global thresholds (full-series based)
merged_window_log['is_above_p95'] = merged_window_log['value'] >= merged_window_log['global_p95']
merged_window_log['is_below_p05'] = merged_window_log['value'] <= merged_window_log['global_p05']

# 5) Aggregate per (cmdb_id, log_name) within the window
total_points = merged_window_log.groupby(['cmdb_id', 'log_name']).size().rename('total_points_in_window').reset_index()
anom_above = merged_window_log[merged_window_log['is_above_p95']].groupby(['cmdb_id', 'log_name'])['timestamp'] \
    .agg(anomaly_count_above_p95=('count')).reset_index()
anom_below = merged_window_log[merged_window_log['is_below_p05']].groupby(['cmdb_id', 'log_name'])['timestamp'] \
    .agg(anomaly_count_below_p05=('count')).reset_index()

earliest_above = merged_window_log[merged_window_log['is_above_p95']].groupby(['cmdb_id', 'log_name'])['timestamp'] \
    .min().rename('earliest_timestamp_above_p95_in_window').reset_index()
earliest_below = merged_window_log[merged_window_log['is_below_p05']].groupby(['cmdb_id', 'log_name'])['timestamp'] \
    .min().rename('earliest_timestamp_below_p05_in_window').reset_index()

# 6) Combine results starting from thresholds to include all series
summary_log = thresholds_log_df.merge(total_points, on=['cmdb_id', 'log_name'], how='left') \
    .merge(anom_above, on=['cmdb_id', 'log_name'], how='left') \
    .merge(anom_below, on=['cmdb_id', 'log_name'], how='left') \
    .merge(earliest_above, on=['cmdb_id', 'log_name'], how='left') \
    .merge(earliest_below, on=['cmdb_id', 'log_name'], how='left')

# fill counts with 0
summary_log['total_points_in_window'] = summary_log['total_points_in_window'].fillna(0).astype(int)
summary_log['anomaly_count_above_p95'] = summary_log['anomaly_count_above_p95'].fillna(0).astype(int)
summary_log['anomaly_count_below_p05'] = summary_log['anomaly_count_below_p05'].fillna(0).astype(int)

# helper to convert epoch seconds to ISO string in Asia/Shanghai
def epoch_to_iso_shtz(val):
    if pd.isna(val):
        return None
    return pd.to_datetime(int(val), unit='s', utc=True).tz_convert(tz).isoformat()

summary_log['earliest_timestamp_above_p95_in_window'] = summary_log['earliest_timestamp_above_p95_in_window'].apply(epoch_to_iso_shtz)
summary_log['earliest_timestamp_below_p05_in_window'] = summary_log['earliest_timestamp_below_p05_in_window'].apply(epoch_to_iso_shtz)

# 7) Prepare final result columns and sorting key (sum of anomalies)
summary_log['total_anomalies'] = summary_log['anomaly_count_above_p95'] + summary_log['anomaly_count_below_p05']

result_log_df = summary_log[[
    'cmdb_id', 'log_name', 'global_p95', 'global_p05',
    'total_points_in_window', 'anomaly_count_above_p95', 'anomaly_count_below_p05',
    'earliest_timestamp_above_p95_in_window', 'earliest_timestamp_below_p05_in_window',
    'total_anomalies'
]]

# sort by total_anomalies desc and limit to top 20
result_log_top20 = result_log_df.sort_values(by=['total_anomalies'], ascending=False).head(20).drop(columns=['total_anomalies']).reset_index(drop=True)

# Display compact top-20 summary
result_log_top20

```
Out[6]:
```
```python
# Plain-English summary of log.csv analysis (compact)
summary = (
    "Log analysis (window 2022-03-20 21:30–22:00 UTC+8):\n"
    "- Top returned series are per-pod log.error_count (examples: adservice-0, adservice-1, adservice-2, checkoutservice-0, cartservice-1, ...).\n"
    "- For these series global_p95 == global_p05 == 0.0.\n"
    "- In the incident window each series has 30 points and anomaly counts show: anomaly_count_above_p95 = 30 and anomaly_count_below_p05 = 30. "
    "Earliest anomaly timestamps are 2022-03-20T21:30:00+08:00.\n"
    "- Interpretation: P05 and P95 collapse because the metric values are constant (zeros), so every point meets both >=P95 and <=P05. This is a threshold artifact, not meaningful high/low anomalies.\n"
    "- Recommended next steps: confirm that log.error_count semantics are correct (zero = no errors vs. telemetry issue), inspect raw per-minute values, and use alternative anomaly logic (e.g., require spread between P05 and P95, rolling baselines, or absolute thresholds) to avoid false positives on constant metrics."
)
summary
```

The original code execution output of IPython Kernel is also provided below for reference:

cmdb_id         log_name  global_p95  global_p05  total_points_in_window  anomaly_count_above_p95  anomaly_count_below_p05 earliest_timestamp_above_p95_in_window earliest_timestamp_below_p05_in_window
0               adservice-0  log.error_count         0.0         0.0                      30                       30                       30              2022-03-20T21:30:00+08:00              2022-03-20T21:30:00+08:00
1               adservice-1  log.error_count         0.0         0.0                      30                       30                       30              2022-03-20T21:30:00+08:00              2022-03-20T21:30:00+08:00
2               adservice-2  log.error_count         0.0         0.0                      30                       30                       30              2022-03-20T21:30:00+08:00              2022-03-20T21:30:00+08:00
3         checkoutservice-0  log.error_count         0.0         0.0                      30                       30                       30              2022-03-20T21:30:00+08:00              2022-03-20T21:30:00+08:00
4             cartservice-1  log.error_count         0.0         0.0                      30                       30                       30              2022-03-20T21:30:00+08:00              2022-03-20T21:30:00+08:00
5             cartservice-0  log.error_count         0.0         0.0                      30                       30                       30              2022-03-20T21:30:00+08:00              2022-03-20T21:30:00+08:00
6             cartservice-2  log.error_count         0.0         0.0                      30                       30                       30              2022-03-20T21:30:00+08:00              2022-03-20T21:30:00+08:00
7         currencyservice-1  log.error_count         0.0         0.0                      30                       30                       30              2022-03-20T21:30:00+08:00              2022-03-20T21:30:00+08:00
8            emailservice-2  log.error_count         0.0         0.0                      30                       30                       30              2022-03-20T21:30:00+08:00              2022-03-20T21:30:00+08:00
9            emailservice-0  log.error_count         0.0         0.0                      30                       30                       30              2022-03-20T21:30:00+08:00              2022-03-20T21:30:00+08:00
10        currencyservice-2  log.error_count         0.0         0.0                      30                       30                       30              2022-03-20T21:30:00+08:00              2022-03-20T21:30:00+08:00
11        checkoutservice-1  log.error_count         0.0         0.0                      30                       30                       30              2022-03-20T21:30:00+08:00              2022-03-20T21:30:00+08:00
12        currencyservice-0  log.error_count         0.0         0.0                      30                       30                       30              2022-03-20T21:30:00+08:00              2022-03-20T21:30:00+08:00
13  productcatalogservice-0  log.error_count         0.0         0.0                      30                       30                       30              2022-03-20T21:30:00+08:00              2022-03-20T21:30:00+08:00
14  productcatalogservice-2  log.error_count         0.0         0.0                      30                       30                       30              2022-03-20T21:30:00+08:00              2022-03-20T21:30:00+08:00
15  recommendationservice-0  log.error_count         0.0         0.0                      30                       30                       30              2022-03-20T21:30:00+08:00              2022-03-20T21:30:00+08:00
16  recommendationservice-2  log.error_count         0.0         0.0                      30                       30                       30              2022-03-20T21:30:00+08:00              2022-03-20T21:30:00+08:00
17  recommendationservice-1  log.error_count         0.0         0.0                      30                       30                       30              2022-03-20T21:30:00+08:00              2022-03-20T21:30:00+08:00
18         paymentservice-1  log.error_count         0.0         0.0                      30                       30                       30              2022-03-20T21:30:00+08:00              2022-03-20T21:30:00+08:00
19        shippingservice-1  log.error_count         0.0         0.0                      30                       30                       30              2022-03-20T21:30:00+08:00              2022-03-20T21:30:00+08:00```
```

In [ ]:
# In[7]:

import pandas as pd
import pytz

# reuse existing error_logs_df, tz, and helper if present
try:
    error_logs_df
except NameError:
    error_logs_df = pd.read_csv("error_logs.csv")

try:
    tz
except NameError:
    tz = pytz.timezone('Asia/Shanghai')

# reuse or define epoch -> ISO converter (Asia/Shanghai)
try:
    epoch_to_iso_shtz
except NameError:
    def epoch_to_iso_shtz(val):
        if pd.isna(val):
            return None
        return pd.to_datetime(int(val), unit='s', utc=True).tz_convert(tz).isoformat()

# Ensure types
error_logs_df['timestamp'] = pd.to_numeric(error_logs_df.get('timestamp'), errors='coerce').astype('Int64')
error_logs_df['cmdb_id'] = error_logs_df.get('cmdb_id').astype(str)
error_logs_df['message'] = error_logs_df.get('message').astype(str)

# Filter for the specified frontend pods
frontend_ids = ['frontend-0', 'frontend-1', 'frontend-2']
frontend_logs = error_logs_df[error_logs_df['cmdb_id'].isin(frontend_ids)].dropna(subset=['timestamp', 'cmdb_id', 'message']).copy()

# Aggregate by (cmdb_id, message)
agg_msg = frontend_logs.groupby(['cmdb_id', 'message'])['timestamp'] \
    .agg(total_count=('size'), earliest_ts_num=('min'), latest_ts_num=('max')).reset_index()

# Convert timestamps to ISO strings in Asia/Shanghai and add truncated message (<=200 chars)
agg_msg['earliest_timestamp'] = agg_msg['earliest_ts_num'].apply(lambda x: epoch_to_iso_shtz(int(x)) if pd.notna(x) else None)
agg_msg['latest_timestamp'] = agg_msg['latest_ts_num'].apply(lambda x: epoch_to_iso_shtz(int(x)) if pd.notna(x) else None)
agg_msg['truncated_message'] = agg_msg['message'].str.slice(0, 200)

# Prepare final compact table and sort by total_count desc, limit to top 20
top20_message_agg = agg_msg.sort_values(by='total_count', ascending=False)[
    ['cmdb_id', 'total_count', 'earliest_timestamp', 'latest_timestamp', 'truncated_message']
].head(20).reset_index(drop=True)

# Display compact result
top20_message_agg

```
Out[7]:
```
```python
# Plain-English summary based on the aggregated error_logs results (compact)
try:
    df = top20_message_agg
except NameError:
    # fallback summary if the aggregated dataframe is not available
    summary = (
        "Frontend error aggregation: top messages are repeated warnings like "
        "'severity: warning, message: failed to retrieve...'. Counts (by cmdb_id): "
        "frontend-2: 479, frontend-0: 317, frontend-1: 248. "
        "Earliest overall error observed: frontend-2 at 2022-03-20T21:30:00+08:00. "
        "Action: investigate frontend pods and their upstream calls for retrieval failures."
    )
else:
    # build a concise summary from the dataframe
    rows = []
    for _, r in df.iterrows():
        rows.append(f"{r['cmdb_id']}: {int(r['total_count'])}")
    counts_str = ", ".join(rows)
    # overall earliest timestamp (first row in df has highest counts; compute global earliest)
    overall_earliest_row = df.loc[df['earliest_timestamp'].notna(), ['cmdb_id','earliest_timestamp']].sort_values('earliest_timestamp').head(1)
    if not overall_earliest_row.empty:
        oe_cmdb = overall_earliest_row.iloc[0]['cmdb_id']
        oe_ts = overall_earliest_row.iloc[0]['earliest_timestamp']
        earliest_str = f"Earliest overall error: {oe_cmdb} at {oe_ts}."
    else:
        earliest_str = "No timestamps available."
    example_msg = df.iloc[0]['truncated_message'] if not df.empty else ""
    summary = (
        f"Frontend error aggregation: top message(s) are repeated warnings like '{example_msg}'. "
        f"Counts (by cmdb_id): {counts_str}. {earliest_str} "
        "Action: inspect the frontend pods and the upstream calls reported in these 'failed to retrieve' warnings."
    )

summary
```

The original code execution output of IPython Kernel is also provided below for reference:

cmdb_id  total_count         earliest_timestamp           latest_timestamp                                  truncated_message
0  frontend-2          479  2022-03-20T21:30:00+08:00  2022-03-20T21:59:57+08:00  severity: warning, message: failed to retrieve...
1  frontend-0          317  2022-03-20T21:30:00+08:00  2022-03-20T21:59:55+08:00  severity: warning, message: failed to retrieve...
2  frontend-1          248  2022-03-20T21:30:05+08:00  2022-03-20T21:59:58+08:00  severity: warning, message: failed to retrieve...```
```

In [ ]:
# In[8]:

import pandas as pd
import pytz

# reuse existing error_logs_df and tz if available, else load/define
try:
    error_logs_df
except NameError:
    error_logs_df = pd.read_csv("error_logs.csv")

try:
    tz
except NameError:
    tz = pytz.timezone('Asia/Shanghai')

# ensure types and drop incomplete rows
error_logs_df['timestamp'] = pd.to_numeric(error_logs_df.get('timestamp'), errors='coerce').astype('Int64')
error_logs_df['cmdb_id'] = error_logs_df.get('cmdb_id').astype(str)
error_logs_df['message'] = error_logs_df.get('message').astype(str)
error_logs_df = error_logs_df.dropna(subset=['timestamp', 'cmdb_id', 'message']).reset_index(drop=True)

# filter for specified frontend pods
frontend_ids = ['frontend-2', 'frontend-0', 'frontend-1']
frontend_logs = error_logs_df[error_logs_df['cmdb_id'].isin(frontend_ids)].copy()

# aggregate by (cmdb_id, message)
agg = frontend_logs.groupby(['cmdb_id', 'message'])['timestamp'] \
    .agg(total_count=('size'), earliest_ts_num=('min'), latest_ts_num=('max')).reset_index()

# helper to convert epoch seconds to ISO string in Asia/Shanghai
def epoch_to_iso_shtz(val):
    if pd.isna(val):
        return None
    return pd.to_datetime(int(val), unit='s', utc=True).tz_convert(tz).isoformat()

# convert timestamps and create truncated message (<=500 chars) for bounded display
agg['earliest_timestamp'] = agg['earliest_ts_num'].apply(lambda x: epoch_to_iso_shtz(int(x)) if pd.notna(x) else None)
agg['latest_timestamp'] = agg['latest_ts_num'].apply(lambda x: epoch_to_iso_shtz(int(x)) if pd.notna(x) else None)
agg['message_truncated'] = agg['message'].str.slice(0, 500)

# prepare final compact table and sort by total_count desc, limit top 20
top20_messages = agg.sort_values(by='total_count', ascending=False)[
    ['cmdb_id', 'total_count', 'earliest_timestamp', 'latest_timestamp', 'message_truncated']
].head(20).reset_index(drop=True)

# find single earliest timestamp among these messages and corresponding cmdb_id and full message (truncated to 500 chars)
if not agg.empty:
    idx = agg['earliest_ts_num'].astype(int).idxmin()
    row = agg.loc[idx]
    overall_earliest_ts_iso = epoch_to_iso_shtz(int(row['earliest_ts_num']))
    overall_earliest_cmdb = row['cmdb_id']
    overall_earliest_msg_trunc = row['message_truncated']
    earliest_note = f"Earliest message among frontends: cmdb_id={overall_earliest_cmdb}, timestamp={overall_earliest_ts_iso}, message_truncated=\"{overall_earliest_msg_trunc}\""
else:
    earliest_note = "No messages found for the specified frontends."

# Display compact results
top20_messages, earliest_note

```
Out[8]:
```
```python
# Compact plain-English summary using previous aggregation results
try:
    df = top20_messages
    note = earliest_note
except NameError:
    # fallback message if variables are not present
    summary = ("No aggregated results available in the kernel. Please re-run the aggregation step on error_logs.csv "
               "for frontend-0, frontend-1, frontend-2 and then request a summary.")
else:
    # Build concise counts string for the top frontend pods (bounded to available rows)
    counts = ", ".join(f"{row['cmdb_id']}: {int(row['total_count'])}" for _, row in df.iterrows())
    example_msg = df.iloc[0]['message_truncated'] if not df.empty else ""
    summary = (
        "Frontend error message summary:\n"
        f"- Top aggregated message (truncated): \"{example_msg}\"\n"
        f"- Counts by frontend pod (top results): {counts}\n"
        f"- {note}\n"
        "Interpretation: frontends report repeated warning-level messages like 'failed to retrieve...'. "
        "Action: investigate these frontend pods and the upstream retrieval calls referenced in the messages."
    )

summary
```

The original code execution output of IPython Kernel is also provided below for reference:

(      cmdb_id  total_count         earliest_timestamp           latest_timestamp                                  message_truncated
0  frontend-2          479  2022-03-20T21:30:00+08:00  2022-03-20T21:59:57+08:00  severity: warning, message: failed to retrieve...
1  frontend-0          317  2022-03-20T21:30:00+08:00  2022-03-20T21:59:55+08:00  severity: warning, message: failed to retrieve...
2  frontend-1          248  2022-03-20T21:30:05+08:00  2022-03-20T21:59:58+08:00  severity: warning, message: failed to retrieve..., 'Earliest message among frontends: cmdb_id=frontend-0, timestamp=2022-03-20T21:30:00+08:00, message_truncated="severity: warning, message: failed to retrieve ads"')```
```

In [ ]:
# In[9]:

import pandas as pd
import pytz
from datetime import datetime

# reuse existing error_logs_df and tz if available
try:
    error_logs_df
except NameError:
    error_logs_df = pd.read_csv("error_logs.csv")

try:
    tz
except NameError:
    tz = pytz.timezone('Asia/Shanghai')

# ensure types
error_logs_df['timestamp'] = pd.to_numeric(error_logs_df.get('timestamp'), errors='coerce').astype('Int64')
error_logs_df['cmdb_id'] = error_logs_df.get('cmdb_id').astype(str)
error_logs_df['message'] = error_logs_df.get('message').astype(str)
error_logs_df = error_logs_df.dropna(subset=['timestamp', 'cmdb_id', 'message']).reset_index(drop=True)

# helper to convert epoch seconds to ISO string in Asia/Shanghai
def epoch_to_iso_shtz(val):
    if pd.isna(val):
        return None
    return pd.to_datetime(int(val), unit='s', utc=True).tz_convert(tz).isoformat()

# filter for specified frontends and the 21:30:00 to 21:31:00 UTC+8 window (inclusive start, exclusive end)
frontend_ids = ['frontend-2', 'frontend-0', 'frontend-1']
start_local = tz.localize(datetime(2022, 3, 20, 21, 30, 0))
end_local = tz.localize(datetime(2022, 3, 20, 21, 31, 0))
start_ts = int(start_local.astimezone(pytz.UTC).timestamp())
end_ts = int(end_local.astimezone(pytz.UTC).timestamp())

filtered_window = error_logs_df[
    (error_logs_df['cmdb_id'].isin(frontend_ids)) &
    (error_logs_df['timestamp'] >= start_ts) &
    (error_logs_df['timestamp'] < end_ts)
].copy()

# aggregate by full message text and cmdb_id
agg_msgs = filtered_window.groupby(['cmdb_id', 'message'])['timestamp'] \
    .agg(total_count=('size'), earliest_ts_num=('min'), latest_ts_num=('max')).reset_index()

# convert timestamps to ISO and prepare full message (truncate display to 500 chars if longer)
agg_msgs['earliest_timestamp'] = agg_msgs['earliest_ts_num'].apply(lambda x: epoch_to_iso_shtz(int(x)) if pd.notna(x) else None)
agg_msgs['latest_timestamp'] = agg_msgs['latest_ts_num'].apply(lambda x: epoch_to_iso_shtz(int(x)) if pd.notna(x) else None)
agg_msgs['message_display'] = agg_msgs['message'].apply(lambda m: m if len(m) <= 500 else m[:500])

# prepare final compact table sorted by total_count desc, limit 20
top20_messages_window = agg_msgs.sort_values(by='total_count', ascending=False)[
    ['cmdb_id', 'message_display', 'total_count', 'earliest_timestamp', 'latest_timestamp']
].head(20).reset_index(drop=True)

# find single earliest message among these filtered rows (based on timestamp)
if not filtered_window.empty:
    idx_earliest = int(filtered_window['timestamp'].astype(int).idxmin())
    row_earliest = filtered_window.loc[idx_earliest]
    earliest_cmdb = row_earliest['cmdb_id']
    earliest_msg_full = row_earliest['message']
    earliest_msg_display = earliest_msg_full if len(earliest_msg_full) <= 500 else earliest_msg_full[:500]
    earliest_ts_iso = epoch_to_iso_shtz(int(row_earliest['timestamp']))
    earliest_note = f"Earliest message: cmdb_id={earliest_cmdb}, timestamp={earliest_ts_iso}, message_truncated=\"{earliest_msg_display}\""
else:
    earliest_note = "No messages in the specified window for the given frontends."

# Display compact results (DataFrame and one-line note)
top20_messages_window, earliest_note

```
Out[9]:
```
```python
# Plain-English summary (compact)
summary = (
    "Window 2022-03-20 21:30:00–21:31:00 (UTC+8) for frontends:\n"
    "- Top messages by count (aggregated per full message):\n"
    "  * frontend-2: 18 occurrences\n"
    "  * frontend-0: 16 occurrences\n"
    "  * frontend-1: 15 occurrences\n"
    "- The dominant message (truncated) is: \"severity: warning, message: failed to retrieve ads\".\n"
    "- Earliest observed among these rows: frontend-2 at 2022-03-20T21:30:00+08:00.\n"
    "Recommendation: investigate the frontend pods' retrieval of 'ads' (upstream calls/endpoints, network errors, or dependent service failures) and check related traces/metrics for those timestamps."
)
summary
```

The original code execution output of IPython Kernel is also provided below for reference:

(      cmdb_id                                    message_display  total_count         earliest_timestamp           latest_timestamp
0  frontend-2  severity: warning, message: failed to retrieve...           18  2022-03-20T21:30:00+08:00  2022-03-20T21:30:57+08:00
1  frontend-0  severity: warning, message: failed to retrieve...           16  2022-03-20T21:30:00+08:00  2022-03-20T21:30:55+08:00
2  frontend-1  severity: warning, message: failed to retrieve...           15  2022-03-20T21:30:05+08:00  2022-03-20T21:30:52+08:00, 'Earliest message: cmdb_id=frontend-2, timestamp=2022-03-20T21:30:00+08:00, message_truncated="severity: warning, message: failed to retrieve ads"')```
```

In [ ]:
# In[10]:

import pandas as pd
import pytz
from datetime import datetime

# reuse existing trace_df if present, else load
try:
    trace_df
except NameError:
    trace_df = pd.read_csv("trace.csv")

# ensure dtypes
trace_df['timestamp'] = pd.to_numeric(trace_df['timestamp'], errors='coerce').astype('Int64')
trace_df['value'] = pd.to_numeric(trace_df['value'], errors='coerce')
trace_df['cmdb_id'] = trace_df['cmdb_id'].astype(str)
trace_df['trace_name'] = trace_df['trace_name'].astype(str)

# timezone and window boundaries (UTC+8)
tz = pytz.timezone('Asia/Shanghai')
start_local = tz.localize(datetime(2022, 3, 20, 21, 30, 0))
end_local = tz.localize(datetime(2022, 3, 20, 21, 31, 0))
start_ts = int(start_local.astimezone(pytz.UTC).timestamp())
end_ts = int(end_local.astimezone(pytz.UTC).timestamp())

# filter criteria
cmdb_filter = ['adservice-0', 'adservice-1', 'adservice-2']
name_mask = trace_df['trace_name'].str.contains('from_frontend', na=False)

window_mask = (trace_df['timestamp'] >= start_ts) & (trace_df['timestamp'] < end_ts)
cmdb_mask = trace_df['cmdb_id'].isin(cmdb_filter)

adservice_frontend_window = trace_df[cmdb_mask & name_mask & window_mask][
    ['timestamp', 'cmdb_id', 'trace_name', 'value']
].sort_values(by=['cmdb_id', 'timestamp'], ascending=[True, True]).head(20).reset_index(drop=True)

adservice_frontend_window

```
Out[10]:
```
Summary (plain English):

- At the incident start (2022-03-20 21:30:00 UTC+8), the adservice pod traces (adservice-0 and adservice-1 shown) have normal-looking values:
  - duration_mean and duration_p95 are very small and stable (~0.00001–0.00002),
  - error_rate = 0.0,
  - row_count values are modest (around 15–18).
- All sampled trace rows at 21:30:00 show no obvious spikes or non-zero error rates — i.e., these adservice traces do not show anomalies coincident with the frontend "failed to retrieve ads" warnings.
- Next steps: inspect frontend-to-adservice network logs, adservice access/error logs, and upstream dependency traces (or adservice-2 rows not shown) to find causes; also check request routing and network connectivity between frontends and adservice.

The original code execution output of IPython Kernel is also provided below for reference:

timestamp      cmdb_id                           trace_name      value
0   1647783000  adservice-0  trace.from_frontend-0.duration_mean   0.000013
1   1647783000  adservice-0   trace.from_frontend-0.duration_p95   0.000020
2   1647783000  adservice-0     trace.from_frontend-0.error_rate   0.000000
3   1647783000  adservice-0      trace.from_frontend-0.row_count  16.000000
4   1647783000  adservice-0  trace.from_frontend-1.duration_mean   0.000012
5   1647783000  adservice-0   trace.from_frontend-1.duration_p95   0.000012
6   1647783000  adservice-0     trace.from_frontend-1.error_rate   0.000000
7   1647783000  adservice-0      trace.from_frontend-1.row_count  15.000000
8   1647783000  adservice-0  trace.from_frontend-2.duration_mean   0.000015
9   1647783000  adservice-0   trace.from_frontend-2.duration_p95   0.000022
10  1647783000  adservice-0     trace.from_frontend-2.error_rate   0.000000
11  1647783000  adservice-0      trace.from_frontend-2.row_count  18.000000
12  1647783000  adservice-1  trace.from_frontend-0.duration_mean   0.000012
13  1647783000  adservice-1   trace.from_frontend-0.duration_p95   0.000015
14  1647783000  adservice-1     trace.from_frontend-0.error_rate   0.000000
15  1647783000  adservice-1      trace.from_frontend-0.row_count  16.000000
16  1647783000  adservice-1  trace.from_frontend-1.duration_mean   0.000012
17  1647783000  adservice-1   trace.from_frontend-1.duration_p95   0.000013
18  1647783000  adservice-1     trace.from_frontend-1.error_rate   0.000000
19  1647783000  adservice-1      trace.from_frontend-1.row_count  16.000000```
```

In [ ]:
# In[11]:

# Reuse existing trace_df and tz if available
try:
    trace_df
except NameError:
    import pandas as pd
    trace_df = pd.read_csv("trace.csv")

import pytz
from datetime import datetime

# ensure timezone (UTC+8)
tz = pytz.timezone('Asia/Shanghai')

# ensure dtypes (reuse existing conversions where possible)
trace_df['timestamp'] = pd.to_numeric(trace_df['timestamp'], errors='coerce').astype('Int64')
trace_df['value'] = pd.to_numeric(trace_df['value'], errors='coerce')
trace_df['cmdb_id'] = trace_df['cmdb_id'].astype(str)
trace_df['trace_name'] = trace_df['trace_name'].astype(str)

# define window 2022-03-20 21:30:00 to 2022-03-20 21:31:00 UTC+8
start_local = tz.localize(datetime(2022, 3, 20, 21, 30, 0))
end_local = tz.localize(datetime(2022, 3, 20, 21, 31, 0))
start_ts = int(start_local.astimezone(pytz.UTC).timestamp())
end_ts = int(end_local.astimezone(pytz.UTC).timestamp())

# filter for adservice-related pods (cmdb_id starting with 'adservice') and time window
mask_cmdb = trace_df['cmdb_id'].str.startswith('adservice', na=False)
mask_time = (trace_df['timestamp'] >= start_ts) & (trace_df['timestamp'] < end_ts)

adservice_traces_window = trace_df[mask_cmdb & mask_time][
    ['timestamp', 'cmdb_id', 'trace_name', 'value']
].sort_values(by=['cmdb_id', 'timestamp'], ascending=[True, True]).head(50).reset_index(drop=True)

# Display compact result (up to 50 rows)
adservice_traces_window

```
Out[11]:
```
Summary (plain English):

- I retrieved adservice-related trace rows at 2022-03-20 21:30:00–21:31:00 (UTC+8) for adservice-0, adservice-1, adservice-2 and adservice2-0.
- Observations at the exact incident start (21:30:00):
  - duration_mean and duration_p95 values are very small and stable (~0.00001–0.00002).
  - error_rate is 0.0 for the sampled series.
  - row_count per frontend-origin is modest (about 15–18) with no evident drop.
- Conclusion: adservice instances do not show latency spikes, increased error_rate, or row_count drops at the same timestamp when frontends logged "failed to retrieve ads." The adservice traces look normal at 21:30:00.
- Suggested next steps: focus on frontend-side investigation (network errors, request routing, or upstream ad provider), check subsequent minutes and related network/connection metrics, and correlate with service logs/traces for upstream dependencies.

The original code execution output of IPython Kernel is also provided below for reference:

timestamp       cmdb_id                            trace_name      value
0   1647783000   adservice-0   trace.from_frontend-0.duration_mean   0.000013
1   1647783000   adservice-0    trace.from_frontend-0.duration_p95   0.000020
2   1647783000   adservice-0      trace.from_frontend-0.error_rate   0.000000
3   1647783000   adservice-0       trace.from_frontend-0.row_count  16.000000
4   1647783000   adservice-0   trace.from_frontend-1.duration_mean   0.000012
..         ...           ...                                   ...        ...
35  1647783000   adservice-2       trace.from_frontend-2.row_count  17.000000
36  1647783000  adservice2-0  trace.from_frontend2-0.duration_mean   0.000020
37  1647783000  adservice2-0   trace.from_frontend2-0.duration_p95   0.000023
38  1647783000  adservice2-0     trace.from_frontend2-0.error_rate   0.000000
39  1647783000  adservice2-0      trace.from_frontend2-0.row_count  15.000000

[40 rows x 4 columns]

**Note**: The printed pandas DataFrame is truncated due to its size. Only **10 rows** are displayed, which may introduce observation bias due to the incomplete table. If you want to comprehensively understand the details without bias, please ask Executor using `df.head(X)` to display more rows.```
```

In [ ]:
# In[12]:

import pandas as pd
import pytz
from datetime import datetime

# Reuse metric_df if available, else load
try:
    metric_df
except NameError:
    try:
        metric_df = dfs.get("metric.csv").copy()
    except Exception:
        metric_df = pd.read_csv("metric.csv")

# Ensure dtypes and drop rows missing essential fields for calculations
metric_df['timestamp'] = pd.to_numeric(metric_df.get('timestamp'), errors='coerce').astype('Int64')
metric_df['value'] = pd.to_numeric(metric_df.get('value'), errors='coerce')
metric_df['cmdb_id'] = metric_df.get('cmdb_id').astype(str)
metric_df['kpi_name'] = metric_df.get('kpi_name').astype(str)
metric_df = metric_df.dropna(subset=['timestamp', 'cmdb_id', 'kpi_name', 'value']).reset_index(drop=True)

# Timezone and incident window (UTC+8)
tz = pytz.timezone('Asia/Shanghai')
start_local = tz.localize(datetime(2022, 3, 20, 21, 30, 0))
end_local = tz.localize(datetime(2022, 3, 20, 22, 0, 0))
start_ts = int(start_local.astimezone(pytz.UTC).timestamp())
end_ts = int(end_local.astimezone(pytz.UTC).timestamp())

# Focus on the requested frontend cmdb_ids
frontend_ids = ['frontend-0', 'frontend-1', 'frontend-2']

# 1) Compute global thresholds (P95 and P05) per (cmdb_id, kpi_name) using full series (before window filtering)
grp_full = metric_df.groupby(['cmdb_id', 'kpi_name'])['value']
global_p95 = grp_full.quantile(0.95).rename('global_p95')
global_p05 = grp_full.quantile(0.05).rename('global_p05')
thresholds_frontends = pd.concat([global_p95, global_p05], axis=1).reset_index()

# Keep only frontend cmdb_ids
thresholds_frontends = thresholds_frontends[thresholds_frontends['cmdb_id'].isin(frontend_ids)].reset_index(drop=True)

# 2) Filter metric points to the incident window and frontends
window_df = metric_df[
    (metric_df['cmdb_id'].isin(frontend_ids)) &
    (metric_df['timestamp'] >= start_ts) &
    (metric_df['timestamp'] < end_ts)
].copy()

# 3) Merge window data with precomputed global thresholds
merged_window = window_df.merge(thresholds_frontends, on=['cmdb_id', 'kpi_name'], how='left')

# 4) Flag anomalies using global thresholds
merged_window['is_above_p95'] = merged_window['value'] >= merged_window['global_p95']
merged_window['is_below_p05'] = merged_window['value'] <= merged_window['global_p05']

# 5) Aggregations within window per (cmdb_id, kpi_name)
total_points = merged_window.groupby(['cmdb_id', 'kpi_name']).size().rename('total_points_in_window').reset_index()
anom_above = merged_window[merged_window['is_above_p95']].groupby(['cmdb_id', 'kpi_name'])['timestamp'] \
    .agg(anomaly_count_above_p95=('count')).reset_index()
anom_below = merged_window[merged_window['is_below_p05']].groupby(['cmdb_id', 'kpi_name'])['timestamp'] \
    .agg(anomaly_count_below_p05=('count')).reset_index()

earliest_above = merged_window[merged_window['is_above_p95']].groupby(['cmdb_id', 'kpi_name'])['timestamp'] \
    .min().rename('earliest_timestamp_above_p95_in_window').reset_index()
earliest_below = merged_window[merged_window['is_below_p05']].groupby(['cmdb_id', 'kpi_name'])['timestamp'] \
    .min().rename('earliest_timestamp_below_p05_in_window').reset_index()

# 6) Combine starting from thresholds to include all frontend KPI series
summary = thresholds_frontends.merge(total_points, on=['cmdb_id', 'kpi_name'], how='left') \
    .merge(anom_above, on=['cmdb_id', 'kpi_name'], how='left') \
    .merge(anom_below, on=['cmdb_id', 'kpi_name'], how='left') \
    .merge(earliest_above, on=['cmdb_id', 'kpi_name'], how='left') \
    .merge(earliest_below, on=['cmdb_id', 'kpi_name'], how='left')

# 7) Fill NaNs for counts with 0 and convert earliest timestamps to ISO (Asia/Shanghai) or None
summary['total_points_in_window'] = summary['total_points_in_window'].fillna(0).astype(int)
summary['anomaly_count_above_p95'] = summary.get('anomaly_count_above_p95', pd.Series()).fillna(0).astype(int) \
    if 'anomaly_count_above_p95' in summary.columns else 0
summary['anomaly_count_below_p05'] = summary.get('anomaly_count_below_p05', pd.Series()).fillna(0).astype(int) \
    if 'anomaly_count_below_p05' in summary.columns else 0

def to_iso_ts(val):
    if pd.isna(val):
        return None
    return pd.to_datetime(int(val), unit='s', utc=True).tz_convert(tz).isoformat()

summary['earliest_timestamp_above_p95_in_window'] = summary['earliest_timestamp_above_p95_in_window'].apply(lambda x: to_iso_ts(x))
summary['earliest_timestamp_below_p05_in_window'] = summary['earliest_timestamp_below_p05_in_window'].apply(lambda x: to_iso_ts(x))

# 8) Sorting key and final selection of columns
summary['total_anomalies'] = summary['anomaly_count_above_p95'] + summary['anomaly_count_below_p05']

result = summary[[
    'cmdb_id', 'kpi_name', 'global_p95', 'global_p05',
    'total_points_in_window', 'anomaly_count_above_p95', 'anomaly_count_below_p05',
    'earliest_timestamp_above_p95_in_window', 'earliest_timestamp_below_p05_in_window',
    'total_anomalies'
]]

# Top 20 by total anomalies (desc). Keep output compact and limited to requested columns.
top20_frontend_metrics = result.sort_values(by=['total_anomalies'], ascending=False).head(20).drop(columns=['total_anomalies']).reset_index(drop=True)

# Display compact top-20 summary
top20_frontend_metrics

```
Out[12]:
```
```python
# Plain-English summary of frontend metric analysis (compact)
summary = (
    "Frontend metrics summary (frontend-0/1/2, window 2022-03-20 21:30–22:00 UTC+8):\n"
    "- The top metric series returned are mesh/istio request/response counters and container spec/state KPIs (examples: mesh.source.frontend.*, mesh.destination.*.istio_request*, container.node-5.container_*).\n"
    "- For these frontend KPIs the global P95 equals the global P05 (many are 0.0 or fixed constants like 65536 or 100000). As a result, every point in the window (30 points) is counted both >=P95 and <=P05: anomaly_count_above_p95 = 30 and anomaly_count_below_p05 = 30, with earliest timestamps at 2022-03-20T21:30:00+08:00.\n"
    "- Interpretation: this is an artifact of collapsed thresholds (constant telemetry values), not meaningful high/low anomalies.\n"
    "- Recommended next steps: inspect raw value distributions for these KPIs to confirm they are constant, verify telemetry/instrumentation, adapt anomaly rules (require P95>P05 or use rolling baselines/absolute thresholds), and continue investigating frontend 'failed to retrieve' warnings (logs/traces and networking/istio) rather than relying on these threshold results alone."
)

summary
```

The original code execution output of IPython Kernel is also provided below for reference:

cmdb_id                                           kpi_name  global_p95  global_p05  total_points_in_window  anomaly_count_above_p95  anomaly_count_below_p05 earliest_timestamp_above_p95_in_window earliest_timestamp_below_p05_in_window
0   frontend-2  mesh.source.frontend.productcatalogservice.ist...         0.0         0.0                      30                       30                       30              2022-03-20T21:30:00+08:00              2022-03-20T21:30:00+08:00
1   frontend-2  mesh.source.frontend.cartservice.istio_request...         0.0         0.0                      30                       30                       30              2022-03-20T21:30:00+08:00              2022-03-20T21:30:00+08:00
2   frontend-2  mesh.source.frontend.cartservice.istio_request...         0.0         0.0                      30                       30                       30              2022-03-20T21:30:00+08:00              2022-03-20T21:30:00+08:00
3   frontend-2  mesh.source.frontend.cartservice.istio_request...         0.0         0.0                      30                       30                       30              2022-03-20T21:30:00+08:00              2022-03-20T21:30:00+08:00
4   frontend-2  mesh.source.frontend.adservice.istio_request_d...         0.0         0.0                      30                       30                       30              2022-03-20T21:30:00+08:00              2022-03-20T21:30:00+08:00
5   frontend-2  mesh.source.frontend.adservice.istio_requests....         0.0         0.0                      30                       30                       30              2022-03-20T21:30:00+08:00              2022-03-20T21:30:00+08:00
6   frontend-2  mesh.destination.unknown.frontend.istio_respon...         0.0         0.0                      30                       30                       30              2022-03-20T21:30:00+08:00              2022-03-20T21:30:00+08:00
7   frontend-2  mesh.source.frontend.adservice.istio_response_...         0.0         0.0                      30                       30                       30              2022-03-20T21:30:00+08:00              2022-03-20T21:30:00+08:00
8   frontend-2  mesh.destination.unknown.frontend.istio_reques...         0.0         0.0                      30                       30                       30              2022-03-20T21:30:00+08:00              2022-03-20T21:30:00+08:00
9   frontend-2  mesh.source.frontend.adservice.istio_request_b...         0.0         0.0                      30                       30                       30              2022-03-20T21:30:00+08:00              2022-03-20T21:30:00+08:00
10  frontend-0  container.node-5.container_ulimits_soft.max_op...     65536.0     65536.0                      30                       30                       30              2022-03-20T21:30:00+08:00              2022-03-20T21:30:00+08:00
11  frontend-0  container.node-5.container_tasks_state.uninter...         0.0         0.0                      30                       30                       30              2022-03-20T21:30:00+08:00              2022-03-20T21:30:00+08:00
12  frontend-2  mesh.destination.unknown.frontend.istio_reques...         0.0         0.0                      30                       30                       30              2022-03-20T21:30:00+08:00              2022-03-20T21:30:00+08:00
13  frontend-2  mesh.destination.unknown.frontend.istio_reques...         0.0         0.0                      30                       30                       30              2022-03-20T21:30:00+08:00              2022-03-20T21:30:00+08:00
14  frontend-0     container.node-5.container_tasks_state.running         0.0         0.0                      30                       30                       30              2022-03-20T21:30:00+08:00              2022-03-20T21:30:00+08:00
15  frontend-2  mesh.destination.unknown.frontend.istio_respon...         0.0         0.0                      30                       30                       30              2022-03-20T21:30:00+08:00              2022-03-20T21:30:00+08:00
16  frontend-0    container.node-5.container_tasks_state.sleeping         0.0         0.0                      30                       30                       30              2022-03-20T21:30:00+08:00              2022-03-20T21:30:00+08:00
17  frontend-0     container.node-5.container_tasks_state.stopped         0.0         0.0                      30                       30                       30              2022-03-20T21:30:00+08:00              2022-03-20T21:30:00+08:00
18  frontend-0         container.node-5.container_spec_cpu_period    100000.0    100000.0                      30                       30                       30              2022-03-20T21:30:00+08:00              2022-03-20T21:30:00+08:00
19  frontend-0  mesh.destination.unknown.frontend.istio_reques...         0.0         0.0                      30                       30                       30              2022-03-20T21:30:00+08:00              2022-03-20T21:30:00+08:00```
```

In [ ]:
# In[13]:

import pandas as pd
import pytz

# reuse metric_df and tz if available
try:
    metric_df
except NameError:
    metric_df = pd.read_csv("metric.csv")

try:
    tz
except NameError:
    tz = pytz.timezone('Asia/Shanghai')

# ensure dtypes
metric_df['timestamp'] = pd.to_numeric(metric_df.get('timestamp'), errors='coerce').astype('Int64')
metric_df['value'] = pd.to_numeric(metric_df.get('value'), errors='coerce')
metric_df['cmdb_id'] = metric_df.get('cmdb_id').astype(str)
metric_df['kpi_name'] = metric_df.get('kpi_name').astype(str)

# filter to requested frontends
frontend_ids = ['frontend-0', 'frontend-1', 'frontend-2']
front_df = metric_df[metric_df['cmdb_id'].isin(frontend_ids)].copy()

# group to get counts per kpi_name
counts = front_df.groupby('kpi_name').size().rename('point_count').reset_index()

# get one example row per kpi_name (choose earliest timestamp sample for stability)
examples = front_df.sort_values(['kpi_name', 'timestamp'], ascending=[True, True]) \
    .drop_duplicates(subset=['kpi_name'], keep='first')[['kpi_name', 'cmdb_id', 'timestamp', 'value']]

# merge counts with examples
summary = counts.merge(examples, on='kpi_name', how='left')

# convert example timestamp to ISO in Asia/Shanghai for readability
def to_iso_shtz(ts):
    if pd.isna(ts):
        return None
    return pd.to_datetime(int(ts), unit='s', utc=True).tz_convert(tz).isoformat()

summary['example_timestamp'] = summary['timestamp'].apply(lambda x: to_iso_shtz(x) if pd.notna(x) else None)

# select and order columns, sort by count desc, limit to top 200
result_top200 = summary[['kpi_name', 'point_count', 'cmdb_id', 'example_timestamp', 'value']] \
    .rename(columns={'cmdb_id': 'example_cmdb_id', 'value': 'example_value'}) \
    .sort_values(by='point_count', ascending=False).head(200).reset_index(drop=True)

# Compact display: result_top200
result_top200

```
Out[13]:
```
```python
# Compact plain-English summary of frontend KPI inventory (metric.csv)
summary = (
    "Frontend KPI inventory (frontend-0/1/2):\n"
    "- Returned up to 164 unique kpi_name entries (limited to 200). Counts indicate how many data points exist per KPI for these frontends.\n"
    "- Top KPI by point count: container.node-5.container_cpu_cfs_periods with 90 points (example: frontend-0 at 2022-03-20T21:30:00+08:00 -> 160.5).\n"
    "- Several container-level CPU metrics (e.g., cpu_cfs_throttled, cpu_load_average_10s, cpu_system_seconds) also have 90 points — these are high-frequency/container metrics.\n"
    "- Many mesh/istio metrics (mesh.source.frontend.* and mesh.destination.*) have 30 points each (example: mesh.source.frontend.adservice.istio_request_* shown with example value 0.0 at 2022-03-20T21:30:00+08:00). That matches a 30-minute window of per-minute data.\n"
    "- Examples show many metrics with constant or zero values in the sample (e.g., throttled=0.0, many mesh counters = 0.0). This can cause collapsed thresholds when doing anomaly detection.\n\n"
    "Recommendation: pick specific KPI names of interest (e.g., particular mesh request counters or frontend CPU/latency KPIs) to inspect full time series or compute anomalies. If you want, I can return the time series for any listed KPI (or the top N KPIs) for deeper analysis."
)
summary
```

The original code execution output of IPython Kernel is also provided below for reference:

kpi_name  point_count example_cmdb_id          example_timestamp  example_value
0           container.node-5.container_cpu_cfs_periods           90      frontend-0  2022-03-20T21:30:00+08:00        160.500
1    container.node-5.container_cpu_cfs_throttled_p...           90      frontend-0  2022-03-20T21:30:00+08:00          0.000
2    container.node-5.container_cpu_cfs_throttled_s...           90      frontend-0  2022-03-20T21:30:00+08:00          0.000
3      container.node-5.container_cpu_load_average_10s           90      frontend-0  2022-03-20T21:30:00+08:00          0.000
4        container.node-5.container_cpu_system_seconds           90      frontend-0  2022-03-20T21:30:00+08:00          0.155
..                                                 ...          ...             ...                        ...            ...
159  mesh.source.frontend.adservice.istio_request_b...           30      frontend-0  2022-03-20T21:30:00+08:00          0.000
160  mesh.source.frontend.checkoutservice.istio_res...           30      frontend-1  2022-03-20T21:30:00+08:00          0.000
161  mesh.source.frontend.checkoutservice.istio_req...           30      frontend-1  2022-03-20T21:30:00+08:00          0.000
162  mesh.source.frontend.checkoutservice.istio_req...           30      frontend-1  2022-03-20T21:30:00+08:00          0.000
163  mesh.source.frontend.checkoutservice.istio_req...           30      frontend-1  2022-03-20T21:30:00+08:00          0.000

[164 rows x 5 columns]

**Note**: The printed pandas DataFrame is truncated due to its size. Only **10 rows** are displayed, which may introduce observation bias due to the incomplete table. If you want to comprehensively understand the details without bias, please ask Executor using `df.head(X)` to display more rows.```
```

In [ ]:
# In[14]:

import pandas as pd
import pytz

# reuse existing trace_df if available, else load
try:
    trace_df
except NameError:
    trace_df = pd.read_csv("trace.csv")

# timezone UTC+8
tz = pytz.timezone('Asia/Shanghai')

# ensure types and drop incomplete rows
trace_df['timestamp'] = pd.to_numeric(trace_df.get('timestamp'), errors='coerce').astype('Int64')
trace_df['value'] = pd.to_numeric(trace_df.get('value'), errors='coerce')
trace_df['cmdb_id'] = trace_df.get('cmdb_id').astype(str)
trace_df['trace_name'] = trace_df.get('trace_name').astype(str)
trace_df = trace_df.dropna(subset=['timestamp', 'cmdb_id', 'trace_name', 'value']).reset_index(drop=True)

# filter for the requested frontends
frontend_ids = ['frontend-0', 'frontend-1', 'frontend-2']
front_traces = trace_df[trace_df['cmdb_id'].isin(frontend_ids)].copy()

# aggregate: point_count per (trace_name, cmdb_id)
counts = front_traces.groupby(['trace_name', 'cmdb_id']).size().rename('point_count').reset_index()

# example row per (trace_name, cmdb_id): choose earliest timestamp as representative
examples = front_traces.sort_values(['trace_name', 'cmdb_id', 'timestamp'], ascending=[True, True, True]) \
    .drop_duplicates(subset=['trace_name', 'cmdb_id'], keep='first') \
    .loc[:, ['trace_name', 'cmdb_id', 'timestamp', 'value']].rename(
        columns={'cmdb_id': 'example_cmdb_id', 'timestamp': 'example_timestamp', 'value': 'example_value'}
    )

# merge counts with examples
inventory = counts.merge(examples, left_on=['trace_name', 'cmdb_id'], right_on=['trace_name', 'example_cmdb_id'], how='left')

# convert example_timestamp to ISO in Asia/Shanghai for readability
def to_iso_shtz(ts):
    if pd.isna(ts):
        return None
    return pd.to_datetime(int(ts), unit='s', utc=True).tz_convert(tz).isoformat()

inventory['example_timestamp'] = inventory['example_timestamp'].apply(lambda x: to_iso_shtz(x) if pd.notna(x) else None)

# select and order columns, sort by point_count desc, limit to top 200
result_inventory = inventory[[
    'trace_name', 'point_count', 'example_cmdb_id', 'example_timestamp', 'example_value'
]].sort_values(by='point_count', ascending=False).head(200).reset_index(drop=True)

# Display the compact inventory (up to 200 rows)
result_inventory

```
Out[14]:
```
```python
# Plain-English summary of the frontend trace inventory (compact)
summary = (
    "Frontend trace inventory (frontend-0/1/2) — top ~200 trace series returned:\n"
    "- Most trace series have 30 data points (one-per-minute across the 30-minute window), consistent with per-minute telemetry.\n"
    "- Examples of trace series and typical values:\n"
    "  * trace.to_shippingservice-2.row_count (example: frontend-2 -> 4)\n"
    "  * trace.from_root.duration_mean (examples: frontend-0 ~0.038s, frontend-1 ~0.037s, frontend-2 ~0.041s)\n"
    "  * trace.to_adservice-1.row_count (examples: frontend pods -> ~16–18)\n"
    "  * Many trace.error_rate metrics are 0.0 in the samples.\n"
    "- Interpretation: the inventory shows many per-service call traces (to adservice, productcatalogservice, shippingservice, checkoutservice, etc.) with stable, low latencies and mostly zero error rates in the sampled points.\n"
    "- Actionable next steps: to investigate frontend 'failed to retrieve ads' warnings, focus on traces that target adservice (trace.to_adservice* / trace.from_frontend* to adservice), examine per-minute row_count, error_rate and duration time series for spikes or drops, and correlate those with frontend log timestamps and network/istio metrics. If you want, I can retrieve the full time series for a few specific trace_names (e.g., all trace.to_adservice* for frontend pods) to look for transient anomalies."
)
summary
```

The original code execution output of IPython Kernel is also provided below for reference:

trace_name  point_count example_cmdb_id          example_timestamp  example_value
0    trace.to_shippingservice-2.row_count           30      frontend-2  2022-03-20T21:30:00+08:00       4.000000
1           trace.from_root.duration_mean           30      frontend-0  2022-03-20T21:30:00+08:00       0.038488
2           trace.from_root.duration_mean           30      frontend-1  2022-03-20T21:30:00+08:00       0.036862
3           trace.from_root.duration_mean           30      frontend-2  2022-03-20T21:30:00+08:00       0.040598
4            trace.from_root.duration_p95           30      frontend-0  2022-03-20T21:30:00+08:00       0.054954
..                                    ...          ...             ...                        ...            ...
195  trace.to_shippingservice-0.row_count           30      frontend-2  2022-03-20T21:30:00+08:00       5.000000
196       trace.to_adservice-1.error_rate           30      frontend-2  2022-03-20T21:30:00+08:00       0.000000
197        trace.to_adservice-1.row_count           30      frontend-0  2022-03-20T21:30:00+08:00      16.000000
198        trace.to_adservice-1.row_count           30      frontend-1  2022-03-20T21:30:00+08:00      16.000000
199        trace.to_adservice-1.row_count           30      frontend-2  2022-03-20T21:30:00+08:00      18.000000

[200 rows x 5 columns]

**Note**: The printed pandas DataFrame is truncated due to its size. Only **10 rows** are displayed, which may introduce observation bias due to the incomplete table. If you want to comprehensively understand the details without bias, please ask Executor using `df.head(X)` to display more rows.```
```